In [14]:
%pip install -q pandas transformers torch tqdm

Note: you may need to restart the kernel to use updated packages.


Lyrics Emotion Classification Pipeline
=======================================
Input:  analysis_df with columns: rank, artist, title, region,
        spotify_uri, lyrics_in_en, original_lang
Output: analysis_df + 13 emotion score columns + dominant_emotion column

In [15]:
import re
import re
import pandas as pd
from tqdm import tqdm
from transformers import pipeline
from pathlib import Path
from datetime import datetime

DATE_PATH = '2026/03/05'
PROJECT_ROOT = Path.cwd().parent  # go up from notebooks to project root

analysis_df = pd.read_csv(PROJECT_ROOT / "data/processed/lyrics/" / DATE_PATH / "lyrics_in_en.csv")

print(f'Loaded {len(analysis_df)} songs')
print(f'Columns: {list(analysis_df.columns)}')

display(analysis_df.head(3))

Loaded 754 songs
Columns: ['rank', 'artist', 'title', 'region', 'spotify_uri', 'lyrics_in_en', 'original_lang']


,rank,artist,title,region,spotify_uri,lyrics_in_en,original_lang
0,1,"Mr Plata, El Americano 4KT",Las Muñequitas,Colombia,4nJJCRYru4QQakCiUA155f,"(Tell me, are you going to give me what I ask ...",es
1,2,"ARIA VEGA, Ryan Castro",CHÉVERE (premium_remix),Colombia,3CBEVPwR3kUXDoTx1lqFUQ,"ARIA VEGA, Ryan Castro\nThe premium costñita a...",es
2,3,"Ryan Castro, Kapo, Gangsta",LA VILLA,Colombia,2ZyrAym0sRLwt4PhGotHuI,"Kapo, Ryan Castro, Gangsta\nWhat a joke, SOG\n...",es


In [16]:
# Config

EMOTIONS = [
    "love", "angst", "joy", "heartbreak", "empowerment",
    "party", "despair", "hope", "nostalgia", "solitude",
    "passion", "relaxing", "other"
]
 
# How many words per chunk (stay under ~400 tokens)
CHUNK_WORDS = 350
 
# Score threshold: below this, treat as 0 (noise floor for multi-label)
NOISE_FLOOR = 0.10

### Clean lyrics

In [17]:
def clean_lyrics(text: str) -> str:
    """
    Remove structural noise but preserve punctuation and casing.
    Punctuation (! ? ...) and capitalisation carry emotional signal
    for NLI-based classifiers — don't strip them.
    """
    if not isinstance(text, str) or not text.strip():
        return ''
 
    # Remove section headers: [Verse 1], [Chorus], [Bridge] etc.
    text = re.sub(r'\[[^\]]*\]', '', text)
 
    # Remove repetition annotations: (x3), (×2), (2x)
    text = re.sub(r'\([\d×xX]+\)', '', text)
 
    # Remove leading artist/feature credits that sometimes appear
    # at the top of translated lyrics (e.g. "ARIA VEGA, Ryan Castro\n")
    # Only strip first line if it looks like a name list (no verb, short)
    lines = text.strip().splitlines()
    if lines and re.match(r'^[A-Za-z\s,&]+$', lines[0].strip()) and len(lines[0]) < 80:
        lines = lines[1:]
    text = '\n'.join(lines)
 
    # Collapse excessive blank lines but keep single line breaks
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
 
    return text.strip()

def dedupe_lines(text: str) -> str:
    """
    Remove duplicate lines (repeated choruses inflate scores).
    Keeps first occurrence; preserves order.
    """
    seen = set()
    result = []
    for line in text.splitlines():
        key = line.strip().lower()
        if key and key not in seen:
            seen.add(key)
            result.append(line.strip())
    return ' '.join(result)
 
 
def prepare_lyrics(text: str) -> str:
    return dedupe_lines(clean_lyrics(text))

### Chunking

In [18]:
def chunk_text(text: str, chunk_words: int = CHUNK_WORDS) -> list[str]:
    """Split text into word-count chunks with a small overlap."""
    words = text.split()
    if not words:
        return []
    overlap = 30
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_words
        chunks.append(' '.join(words[start:end]))
        start = end - overlap  # small overlap so context isn't lost at boundaries
        if start >= len(words):
            break
    return chunks

### Chunking & Classification

In [19]:
# ─────────────────────────────────────────────
# Chunking long lyrics for NLI classification
# ─────────────────────────────────────────────

def chunk_text(text: str, chunk_words: int = CHUNK_WORDS) -> list[str]:
    """Split text into word-count chunks with a small overlap."""
    words = text.split()
    if not words:
        return []
    overlap = 30
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_words
        chunks.append(' '.join(words[start:end]))
        start = end - overlap  # small overlap so context isn't lost at boundaries
        if start >= len(words):
            break
    return chunks
 
 
# ─────────────────────────────────────────────
# Classification
# ─────────────────────────────────────────────
 
def load_classifier():
    """
    facebook/bart-large-mnli  — best general zero-shot NLI classifier.
    Falls back to a smaller model if memory is tight.
    Switch to cross-encoder/nli-deberta-v3-large for higher accuracy
    at the cost of ~2× slower inference.
    """
    print("Loading zero-shot classifier (bart-large-mnli)...")
    return pipeline(
        "zero-shot-classification",
        model="facebook/bart-large-mnli",
        device=0  # set to -1 to force CPU; 0 for first GPU
    )
 
 
def classify_song(lyrics_prepared: str, classifier) -> dict:
    """
    Classify a single (prepared) lyrics string.
    Chunks long lyrics and averages scores across chunks.
    Returns dict {emotion: score} with scores in [0, 1].
    """
    if not lyrics_prepared.strip():
        return {e: 0.0 for e in EMOTIONS}
 
    chunks = chunk_text(lyrics_prepared)
    all_scores = []
 
    for chunk in chunks:
        result = classifier(
            chunk,
            candidate_labels=EMOTIONS,
            multi_label=True   # critical — a song can be love AND nostalgia
        )
        chunk_scores = dict(zip(result["labels"], result["scores"]))
        all_scores.append(chunk_scores)
 
    # Average across chunks
    avg_scores = {
        e: round(sum(s[e] for s in all_scores) / len(all_scores), 4)
        for e in EMOTIONS
    }
 
    # Apply noise floor
    avg_scores = {e: (v if v >= NOISE_FLOOR else 0.0) for e, v in avg_scores.items()}
 
    return avg_scores

In [ ]:
df =  analysis_df.copy()

 # Step 1: Prepare lyrics
print("Cleaning and deduplicating lyrics...")
df['lyrics_prepared'] = df['lyrics_in_en'].fillna('').map(prepare_lyrics)

has_lyrics = df['lyrics_prepared'].str.strip().ne('')
print(f"Songs with usable lyrics: {has_lyrics.sum()} / {len(df)}")

# Step 2: Load model
classifier = load_classifier()

# Step 3: Classify each song
print(f"\nClassifying {has_lyrics.sum()} songs across {EMOTIONS}...")
results = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying"):
    if not row['lyrics_prepared'].strip():
        scores = {e: 0.0 for e in EMOTIONS}
    else:
        scores = classify_song(row['lyrics_prepared'], classifier)
    results.append(scores)

# Step 4: Merge scores into dataframe
scores_df = pd.DataFrame(results, index=df.index)
scores_df.columns = [f"emotion_{e}" for e in EMOTIONS]
df = pd.concat([df, scores_df], axis=1)

# Step 5: Derive dominant emotion per song
emotion_cols = [f"emotion_{e}" for e in EMOTIONS]
df['dominant_emotion'] = df[emotion_cols].idxmax(axis=1).str.replace('emotion_', '')
df['dominant_emotion'] = df['dominant_emotion'].where(
    df[emotion_cols].max(axis=1) > NOISE_FLOOR, other='unclassified'
)

# Drop working column
df = df.drop(columns=['lyrics_prepared'])

df

Cleaning and deduplicating lyrics...
Songs with usable lyrics: 691 / 754
Loading zero-shot classifier (bart-large-mnli)...


Loading weights: 100%|██████████| 515/515 [00:00<00:00, 5853.39it/s]



Classifying 691 songs across ['love', 'angst', 'joy', 'heartbreak', 'empowerment', 'party', 'despair', 'hope', 'nostalgia', 'solitude', 'passion', 'relaxing', 'other']...


Classifying:  18%|█▊        | 134/754 [4:06:27<6:24:09, 37.18s/it]    